# TROPESS HDO search

Scratch notebook for browsing the TROPESS HDO collections at GES DISC through CMR, with the same listing `tropess_download.py` uses, and for looking inside a file. Nothing here downloads.

In [ ]:
import os
from collections import Counter

import requests
import xarray as xr

from utils.path_utils import load_config
from tropess_download import list_collection, parse_date_range, date_runs

settings = load_config('config/config_download.yaml')
by_name = {c['name']: c for c in settings['collections']}

## Collections

Every TROPESS HDO collection GES DISC lists, to catch one that is new since 2026-10-22 (the handoff's table has 14 to fetch, plus 2 complete on disk).

In [ ]:
# earthaccess.search_datasets does not take a short name pattern, so ask CMR directly
response = requests.get(
    'https://cmr.earthdata.nasa.gov/search/collections.umm_json',
    params={'short_name': 'TRPS*L2HDO*', 'options[short_name][pattern]': 'true',
            'provider': 'GES_DISC', 'page_size': 100},
    timeout=60,
)
configured = {c['concept_id'] for c in settings['collections']}
for c in sorted(response.json()['items'], key=lambda c: (c['umm']['ShortName'], c['umm']['Version'])):
    concept_id = c['meta']['concept-id']
    print(f"{c['umm']['ShortName']}.{c['umm']['Version']:<3} {concept_id:<24} {'configured' if concept_id in configured else ''}")

## Granules of one collection

Layout: `TROPESS_<type>/<ShortName>.<version>/<YYYY>/TROPESS_<platform>_L2_<type>_HDO_<YYYYMMDD>_MUSES_<R1pNN>_<stream>_<F0pN>[_J<n>].nc`, one file per day.

In [ ]:
listing = list_collection(settings, by_name['TRPSDL2HDOCRS1FS.1'], parse_date_range(['2021-04-01', '2024-09-05']))
print(len(listing['granules']), 'granules,', len(listing['held']), 'held in', listing['existing'])
print(Counter((g['algorithm'], g['format']) for g in listing['granules']))
print('not held:', date_runs(g['date'] for g in listing['files']))

## Inside a file

Root: `time` (TAI93 seconds), `latitude`, `longitude`, `x` (δD profile), `pressure`, ..., on dims `target` and `level` (17); groups `geolocation`, `geophysical`, `retrieval`, and `observation_ops` in Standard. Root attributes `ShortName` and `VersionID` name the collection.

In [ ]:
path = os.path.join(
    settings['directories']['download'],
    'AIRS/TRPSYL2HDOAIRSORS_1-20240806_185241/TROPESS_AIRS-Aqua_L2_Summary_HDO_20020901_MUSES_R1p6_RS_F0p8_J0.nc',
)
ds = xr.open_dataset(path)
print(ds.attrs['ShortName'], ds.attrs['VersionID'])
ds